# Baseline Comparison — Lottery Model Lab v1

**Purpose**: Reproduce and validate the core simulation results for Baseline A (N=38 vs N=10,600)  
and the D-3 bracket allocation comparison (D-3A / D-3B / D-3C).

**Gate**: All AC checks pass before any simulation is run. SC-1 and SC-3 are verified inline.

## 0. Assumptions and open items

| Label | Status | Value |
|---|---|---|
| burn_rate | CONFIRMED (SRC-013) | 0.20 |
| bracket_allocations | CONFIRMED (SRC-013) | 2/3/5/10/20/40% |
| inject_amount | ASSUMPTION (sim README) | 8,000 [price_units] |
| inject_schedule | ASSUMPTION | (1,0,1,0,1,0,1) — 4/7 rounds |
| burn_applies_to_injection | ASSUMPTION | True |
| ticket_price | **OPEN** — no confirmed source | 2.0 used as placeholder |
| rollover_mode | ASSUMPTION | global (per_bracket unverified) |
| n_players (A1) | ASSUMPTION | 38 (Round 1949 observation) |
| n_players (A2) | ASSUMPTION | 10,600 (fictional — from prior sim) |
| D-3B/C scaling | ASSUMPTION | raw sums × 0.80 to match burn PPF |

**Monetary values in this notebook are in [price_units]** — the unit of ticket_price.  
Do not interpret as USD or CAKE until ticket_price is confirmed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # research/ is parent of notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## 1. Analytical calibration (AC-1, AC-2, AC-3, AC-4)

These run at import. This cell makes them visible and adds AC-4.

In [ ]:
from constants import BRACKET_PROBS, e_tau, ev_per_ticket_approx, E_TAU_6_AT_N38, E_TAU_6_AT_N10600
from model_config import BASELINE_A, D3_CONFIGS

# AC-1 — probability normalization
assert abs(sum(BRACKET_PROBS) - 0.1) < 1e-9, "AC-1 FAIL"
print(f"AC-1 PASS  sum(BRACKET_PROBS) = {sum(BRACKET_PROBS):.9f}")

# AC-2 — prize pool consistency for all configs used in this notebook
for name, cfg in D3_CONFIGS.items():
    cfg.validate()  # raises AssertionError on failure
    print(f"AC-2 PASS  {name}  alloc_sum={sum(cfg.bracket_allocations):.6f}")

# AC-3 — scale formula calibration
assert 26_000 < E_TAU_6_AT_N38 < 27_000,    "AC-3 FAIL (N=38)"
assert 90 < E_TAU_6_AT_N10600 < 100,         "AC-3 FAIL (N=10600)"
print(f"AC-3 PASS  E[tau_6|N=38]={E_TAU_6_AT_N38:.1f}  E[tau_6|N=10600]={E_TAU_6_AT_N10600:.1f}")

# AC-4 — EV order of magnitude
ev_large = ev_per_ticket_approx(k=1, alpha_k=0.02, prize_pool=100_000, n=10_000)
ev_small = ev_per_ticket_approx(k=1, alpha_k=0.02, prize_pool=100_000, n=38)
assert 2.0 < ev_large < 2.5, f"AC-4 FAIL (large N): {ev_large:.3f}"
assert 500 < ev_small < 700, f"AC-4 FAIL (small N): {ev_small:.3f}"
print(f"AC-4 PASS  EV(N=10000)={ev_large:.3f}  EV(N=38)={ev_small:.3f} [price_units/ticket]")

## 2. SC-3 — Single-round hand-verification

Fix seed=0, N=10, mean_tickets=1, inject_schedule=(0,) (no injection this round).  
Expected: revenue=20, fresh_prize=16, payout+carry_out=16.0 exactly.

In [ ]:
from simulator import simulate
from model_config import RunConfig, SimParams, ModelConfig, TokenBurnEcon

_sc3_model = ModelConfig(
    name="SC-3 check",
    econ=TokenBurnEcon(burn_rate=0.20),
    bracket_allocations=(0.02, 0.03, 0.05, 0.10, 0.20, 0.40),
    rollover_mode="global",
    matching_direction="L",
    inject_amount=0.0,
    inject_schedule=(0,),  # no injection
    burn_applies_to_injection=True,
    ticket_price=2.0,
)

_sc3_cfg = RunConfig(
    model=_sc3_model,
    sim=SimParams(n_players=10, mean_tickets_per_player=1.0, rounds=1, simulations=1, seed=0),
    label="SC-3",
)

sc3 = simulate(_sc3_cfg)
row = sc3.iloc[0]
print(f"revenue={row.revenue:.4f}  fresh_prize=revenue×0.80={row.revenue*0.80:.4f}")
print(f"payout={row.payout:.4f}  carry_out={row.carry_out:.4f}  sum={row.payout+row.carry_out:.4f}")
print(f"prize_pool={row.prize_pool:.4f}")
assert abs(row.payout + row.carry_out - row.prize_pool) < 1e-9, "SC-3 FAIL: payout+carry_out != prize_pool"
print("SC-3 PASS  payout + carry_out == prize_pool")

## 3. A1 vs A2 — Scale reversal check

A1: N=38 (Round 1949 observation).  
A2: N=10,600 (fictional prior assumption).  

Expected result: carry-to-revenue ratio reverses dramatically between A1 and A2.

In [ ]:
from compare import run_set

A1_cfg = RunConfig(model=BASELINE_A, sim=SimParams(n_players=38),      label="A1 (N=38)")
A2_cfg = RunConfig(model=BASELINE_A, sim=SimParams(n_players=10_600),  label="A2 (N=10600)")

scale_table = run_set([A1_cfg, A2_cfg])

cols_display = [
    "carry_ratio_mean", "carry_ratio_p50", "carry_ratio_p95",
    "p_carry_gt_10x_rev",
    "hit_rate_k6", "hit_rate_k6_theo",
    "payout_fraction_mean",
]
scale_table[cols_display].T

In [ ]:
# Scale reversal gate
a1_ratio = scale_table.loc["A1 (N=38)",    "carry_ratio_mean"]
a2_ratio = scale_table.loc["A2 (N=10600)", "carry_ratio_mean"]

assert a1_ratio > a2_ratio * 10, (
    f"Scale reversal not observed: A1 carry_ratio={a1_ratio:.1f}, A2={a2_ratio:.1f}"
)
print(f"Scale reversal CONFIRMED: A1 carry/rev={a1_ratio:.1f}×  A2 carry/rev={a2_ratio:.3f}×")

## 4. SC-1 — Bracket hit frequency convergence

Compare empirical hit rates against theoretical predictions for A1 (N=38) and A2 (N=10,600).

In [ ]:
from constants import p_bracket_pays

sc1_cfg = RunConfig(
    model=BASELINE_A,
    sim=SimParams(n_players=10_000, rounds=100, simulations=1_000, seed=42),
    label="SC-1 (N=10000)",
)
sc1_df = simulate(sc1_cfg)

# Theoretical vs empirical at N=10,000
print("SC-1 at N=10,000:")
print(f"{'Bracket':>8} {'Theoretical':>12} {'Empirical':>12} {'Pass':>6}")
bounds = [(0.995, 1.000), (0.995, 1.000), (0.990, 1.000), (0.55, 0.64), (0.07, 0.11), (0.005, 0.015)]
all_pass = True
for k in range(1, 7):
    emp   = sc1_df[f"hit_k{k}"].mean()
    theo  = p_bracket_pays(k, 10_000)
    lo, hi = bounds[k-1]
    ok    = lo <= emp <= hi
    all_pass = all_pass and ok
    print(f"      k={k}   {theo:>12.4f}   {emp:>12.4f}   {'OK' if ok else 'FAIL':>6}")

if all_pass:
    print("\nSC-1 PASS (N=10,000)")
else:
    print("\nSC-1 FAIL — check winner sampling")

In [ ]:
# SC-1 at N=38
sc1_cfg38 = RunConfig(
    model=BASELINE_A,
    sim=SimParams(n_players=38, rounds=100, simulations=1_000, seed=42),
    label="SC-1 (N=38)",
)
sc1_df38 = simulate(sc1_cfg38)

print("SC-1 at N=38:")
print(f"{'Bracket':>8} {'Theoretical':>12} {'Empirical':>12} {'Acceptable range':>20}")
bounds38 = [(0.95, 0.99), (0.26, 0.35), (0.02, 0.05), (0, 0.01), (0, 0.01), (0, 0.01)]
for k in range(1, 7):
    emp  = sc1_df38[f"hit_k{k}"].mean()
    theo = p_bracket_pays(k, 38)
    lo, hi = bounds38[k-1]
    ok   = lo <= emp <= hi
    print(f"      k={k}   {theo:>12.4f}   {emp:>12.4f}   [{lo:.3f}, {hi:.3f}]   {'OK' if ok else 'FAIL'}")

## 5. D-3 bracket allocation comparison

D-3A = PCS confirmed.  
D-3B = SRC-017 scaled ×0.80 (ASSUMPTION).  
D-3C = SRC-005 scaled ×0.80 (ASSUMPTION).

Run at N=38 (the confirmed scale) and N=10,600 (fictional reference).

In [ ]:
d3_configs_small = [
    RunConfig(model=cfg, sim=SimParams(n_players=38),     label=f"{name} N=38")
    for name, cfg in D3_CONFIGS.items()
]
d3_configs_large = [
    RunConfig(model=cfg, sim=SimParams(n_players=10_600), label=f"{name} N=10600")
    for name, cfg in D3_CONFIGS.items()
]

d3_table = run_set(d3_configs_small + d3_configs_large)

d3_cols = [
    "carry_ratio_mean", "p_carry_gt_10x_rev",
    "hit_rate_k6", "payout_fraction_mean",
    "jackpot_carry_mean", "p_jackpot_hits_by_T",
]
d3_table[d3_cols]

## 6. Carry distribution plots — A1 vs A2

In [ ]:
# Re-run full simulations for plotting (500 rounds, 2000 sims — default SimParams)
A1_df = simulate(A1_cfg)
A2_df = simulate(A2_cfg)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, df, label in [(axes[0], A1_df, "A1 (N=38)"), (axes[1], A2_df, "A2 (N=10600)")]:
    ratio = (df["carry_out"] / df["revenue"].replace(0, np.nan)).dropna()
    ax.hist(ratio, bins=60, color="steelblue", edgecolor="none", alpha=0.8)
    ax.axvline(ratio.mean(), color="firebrick", lw=1.5, label=f"mean={ratio.mean():.1f}")
    ax.set_title(f"carry/revenue — {label}")
    ax.set_xlabel("carry_out / revenue")
    ax.set_ylabel("frequency")
    ax.legend()

plt.suptitle("Carry-to-revenue ratio distribution (all rounds × all sims)", y=1.02)
plt.tight_layout()
plt.show()

## 7. D-3 allocation comparison — carry and jackpot metrics plot

In [ ]:
d3_small = d3_table.loc[[idx for idx in d3_table.index if "N=38" in idx]]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names = [n.replace(" N=38", "") for n in d3_small.index]

axes[0].bar(names, d3_small["carry_ratio_mean"], color="steelblue")
axes[0].set_title("Mean carry / revenue (N=38)")
axes[0].set_ylabel("ratio")

axes[1].bar(names, d3_small["payout_fraction_mean"], color="seagreen")
axes[1].set_title("Mean payout fraction (N=38)")
axes[1].set_ylabel("payout / prize_pool")

axes[2].bar(names, d3_small["p_jackpot_hits_by_T"], color="darkorange")
axes[2].set_title(f"P(jackpot hits at least once in T={A1_cfg.sim.rounds} rounds)")
axes[2].set_ylabel("fraction of sims")

for ax in axes:
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("D-3 allocation comparison — N=38 (ASSUMPTION: scaling ×0.80)", y=1.02)
plt.tight_layout()
plt.show()

## 8. Interpretation

**Scale reversal (A1 vs A2)**  
At N=38 the carry-to-revenue ratio is orders of magnitude larger than at N=10,600.  
This means the prize pool at small scale is dominated by accumulated carry, not by fresh ticket revenue.  
The Round 1949 observation (prize pot ≈ 18,529 CAKE with only 38 players) is consistent with this regime.

**D-3 allocation comparison**  
D-3C (SRC-005, frequency-balanced) shifts weight toward low brackets (k=1–3).  
At N=38 this increases payout frequency but reduces jackpot carry accumulation.  
D-3B (SRC-017) concentrates weight at k=6 further than D-3A, increasing jackpot size at the cost of more rounds with no payout.

**Open items that affect interpretation**  
- `ticket_price` is OPEN — all monetary figures are in [price_units].  
- D-3B/C scaling (×0.80) is an ASSUMPTION — see `model_config.py` for the alternative interpretation.  
- All N values other than 38 are fictional; interpret A2 as a structural contrast only.

**Next steps**  
- EV-1 full form: fetch Round 1948 carry_out and Round 1949 revenue via `frefrik/pancakeswap-lottery`.  
- SC-4: long-run pool stability at N=10,600, T=10,000.  
- Phase 5: SALib sensitivity analysis over (n_players, inject_amount, burn_rate).